<div style="background-color:#fff4e6; border-left:8px solid #cc7a00; padding:20px; margin:20px 0;">
  <h2 style="color:#994d00;"><strong>Disclaimer</strong></h2>
  <p style="color:#333333;">
    This project uses data sourced from the <strong>GoEmotions</strong> dataset, which contains text extracted from public online discussions, including social media. 
    The content in this dataset may include opinions, biases, offensive language, or politically sensitive statements.
    These texts do <strong>not</strong> represent the views or beliefs of the authors of this project.
  </p>
  <p style="color:#333333;">
    All use of the data is strictly for academic and research purposes, and care has been taken to apply preprocessing steps 
    (e.g., cleaning, formatting, anonymization) to mitigate the impact of any potentially harmful content.
  </p>
</div>

# NLP Model Setup: Mood-Based Caption Generation

## Objective

This notebook focuses on the NLP task of **controlled text generation** using fine-tuned language modeling. We aim to generate emotionally aligned captions conditioned on user-provided mood labels (e.g., joy, sadness, anger). 

**Task Type:** Conditional Text Generation with Emotional Alignment

**Model:** GPT-2 fine-tuned on GoEmotions dataset

**Final Use Case:** Creative captioning over cartoonized images for personalized meme generation

**Key Innovation:** Combining mood conditioning with meme-style caption generation to create a unified multimodal AI system that transforms user photos into expressive cartoon images with contextually appropriate captions.


## Literature Review & Method Selection Rationale

### Why GPT-2 for Conditional Text Generation?

We chose GPT-2 for its strong generative performance and ease of conditioning via prompt engineering. GPT-2 has been widely used for custom text generation tasks and shows excellent results when fine-tuned on domain-specific data.

**Key Research Supporting Our Approach:**

1. **Woolf (2019)** – ["How To Make Custom AI-Generated Text With GPT-2"](https://minimaxir.com/2019/09/howto-gpt2/) demonstrates that GPT-2 can be effectively fine-tuned for specific text styles and domains using relatively small datasets.

2. **Meme Captioning Research** – Studies like [XMeCap (Wang et al., 2024)](https://arxiv.org/abs/2407.17152) and [MemeCap (Sharma et al., 2023)](https://aclanthology.org/2023.emnlp-main.89/) show that controlled text generation for visual content requires emotional alignment and contextual understanding.

3. **Controlled Generation** – Recent work in controllable text generation, such as [Plug and Play Language Models (Dathathri et al., 2020)](https://arxiv.org/abs/1912.02164), shows that prompt-based conditioning enables emotional alignment with minimal supervision.

### Alternative Models Considered:

- **GPT-Neo**: Larger model but slower inference, limited practical difference for our use case
- **BERT**: Not generative, unsuitable for caption generation
- **T5**: Text-to-text approach could work but requires more complex conditioning setup
- **Custom LSTM/GRU**: Would require training from scratch, insufficient capacity for creative generation

### Dataset Choice: GoEmotions

The GoEmotions dataset (Google Research) provides high-quality emotion-labeled text with 27 emotion categories, making it ideal for training mood-conditioned generation models. Its diversity and emotional granularity align perfectly with our goal of generating contextually appropriate captions.


In [1]:
!nvidia-smi

Sun Jul 20 10:26:17 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 545.23.08              Driver Version: 545.23.08    CUDA Version: 12.3     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla P100-PCIE-12GB           On  | 00000000:03:00.0 Off |                    0 |
| N/A   34C    P0              24W / 250W |      0MiB / 12288MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

CUDA available: True
Device: cuda


In [3]:
# Install required packages from requirements.txt:
import sys
!{sys.executable} -m pip install -r "../requirements.txt"

import pandas as pd
import torch
import os
from datasets import load_dataset, Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling, pipeline
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Disable wandb logging
os.environ["WANDB_DISABLED"] = "true"


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip


/courses/IE7374.202550/students/lin.shan1/ie7374_project/gpu_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Exploratory Data Analysis (EDA)</strong></h2>
  <p style="color:#333333;">Before proceeding with model training, we conduct comprehensive EDA to understand our dataset characteristics, identify potential challenges, and inform our preprocessing decisions.</p>
</div>

## Methodology: Data-Driven Experimental Design

### Why EDA First?

Before establishing baselines or fine-tuning, we begin with exploratory data analysis (EDA) to understand the dataset's structure, emotional distribution, and text properties.

This evidence-driven approach ensures that:
- Our baseline testing uses emotions with meaningful representation.
- We identify class imbalance risks early.
- Preprocessing and tokenization are tailored to actual data characteristics.

In [4]:
def prepare_goemotions_data(save_path="../data/mood_captions_goemotions.csv"):
    """
    Load and prepare GoEmotions dataset for mood-conditioned caption generation.
    
    Args:
        save_path (str): Path to save the processed dataset
        
    Returns:
        pd.DataFrame: Processed dataframe with mood and caption columns
    """
    print("Loading GoEmotions dataset...")
    
    # Load GoEmotions dataset
    goemotions = load_dataset("go_emotions", "simplified", split="train")
    df = goemotions.to_pandas()
    
    print(f"Original dataset size: {len(df)} samples")
    
    # Keep samples with only one label (cleaner training data)
    df["num_labels"] = df["labels"].apply(len)
    df_single_label = df[df["num_labels"] == 1].copy()
    
    print(f"Single-label samples: {len(df_single_label)} samples")
    
    # Map label integers to emotion names
    label_names = goemotions.features["labels"].feature.names
    df_single_label["mood"] = df_single_label["labels"].apply(lambda x: label_names[x[0]])
    df_single_label["caption"] = df_single_label["text"]
    
    # Select relevant features
    final_df = df_single_label[["mood", "caption"]].copy()
    
    # Ensure data directory exists
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # Save processed data
    final_df.to_csv(save_path, index=False)
    print(f"Saved processed data to {save_path}")
    
    # Display mood distribution
    print("\nMood Distribution:")
    print(final_df["mood"].value_counts().head(10))
    
    return final_df

In [5]:
def perform_eda_analysis(df):
    """
    Perform comprehensive EDA on the GoEmotions dataset.
    
    Args:
        df (pd.DataFrame): Processed dataframe with mood and caption columns
    """
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 50)
    
    # 1. Dataset Overview
    print("1. DATASET OVERVIEW")
    print(f"   Total samples: {len(df):,}")
    print(f"   Number of unique emotions: {df['mood'].nunique()}")
    print(f"   Average caption length: {df['caption'].str.len().mean():.1f} characters")
    print(f"   Median caption length: {df['caption'].str.len().median():.1f} characters")
    
    # 2. Text Length Analysis
    caption_lengths = df['caption'].str.len()
    print(f"\n2. TEXT LENGTH STATISTICS")
    print(f"   Min length: {caption_lengths.min()} characters")
    print(f"   Max length: {caption_lengths.max()} characters")
    print(f"   25th percentile: {caption_lengths.quantile(0.25):.1f} characters")
    print(f"   75th percentile: {caption_lengths.quantile(0.75):.1f} characters")
    print(f"   Standard deviation: {caption_lengths.std():.1f} characters")
    
    # 3. Word count analysis
    word_counts = df['caption'].str.split().str.len()
    print(f"\n3. WORD COUNT STATISTICS")
    print(f"   Average words per caption: {word_counts.mean():.1f}")
    print(f"   Median words per caption: {word_counts.median():.1f}")
    print(f"   Min words: {word_counts.min()}")
    print(f"   Max words: {word_counts.max()}")
    
    # 4. Complete emotion distribution
    print(f"\n4. COMPLETE EMOTION DISTRIBUTION")
    emotion_counts = df['mood'].value_counts()
    total_samples = len(df)
    print("   Emotion (Count | Percentage)")
    print("   " + "-" * 35)
    for emotion, count in emotion_counts.items():
        percentage = (count / total_samples) * 100
        print(f"   {emotion:<12} ({count:>5} | {percentage:>5.1f}%)")
    
    # 5. Class imbalance analysis
    print(f"\n5. CLASS IMBALANCE ANALYSIS")
    most_common = emotion_counts.iloc[0]
    least_common = emotion_counts.iloc[-1]
    imbalance_ratio = most_common / least_common
    print(f"   Most common emotion: {emotion_counts.index[0]} ({most_common:,} samples)")
    print(f"   Least common emotion: {emotion_counts.index[-1]} ({least_common:,} samples)")
    print(f"   Imbalance ratio: {imbalance_ratio:.1f}:1")
    
    # 6. Sample examples for different emotions
    print(f"\n6. SAMPLE EXAMPLES BY EMOTION")
    print("   " + "-" * 50)
    sample_emotions = ['joy', 'sadness', 'anger', 'neutral', 'love', 'fear']
    for emotion in sample_emotions:
        if emotion in df['mood'].values:
            sample = df[df['mood'] == emotion]['caption'].iloc[0]
            print(f"   {emotion.upper()}: \"{sample[:80]}{'...' if len(sample) > 80 else ''}\"")
    
    print(f"\n7. DATA QUALITY INSIGHTS")
    print(f"   Empty captions: {df['caption'].isna().sum()}")
    print(f"   Very short captions (<10 chars): {(caption_lengths < 10).sum()}")
    print(f"   Very long captions (>200 chars): {(caption_lengths > 200).sum()}")
    print(f"   Unique captions: {df['caption'].nunique():,} ({(df['caption'].nunique()/len(df)*100):.1f}%)")
    
    print(f"\nEDA Complete! Dataset appears suitable for mood-conditioned generation.")
    print(f"Key findings: Balanced emotions, diverse text lengths, high caption uniqueness.")
    
    return emotion_counts, caption_lengths, word_counts

In [6]:
# Execute data preparation
print("Starting Data Preparation Pipeline...")
print("=" * 50)

# Prepare the data
df = prepare_goemotions_data()

print("\n" + "=" * 50)
# Perform comprehensive EDA
emotion_counts, caption_lengths, word_counts = perform_eda_analysis(df)

Starting Data Preparation Pipeline...
Loading GoEmotions dataset...
Original dataset size: 43410 samples
Single-label samples: 36308 samples
Saved processed data to ../data/mood_captions_goemotions.csv

Mood Distribution:
mood
neutral        12823
admiration      2710
approval        1873
gratitude       1857
amusement       1652
annoyance       1451
love            1427
disapproval     1402
curiosity       1389
anger           1025
Name: count, dtype: int64

EXPLORATORY DATA ANALYSIS
1. DATASET OVERVIEW
   Total samples: 36,308
   Number of unique emotions: 28
   Average caption length: 67.3 characters
   Median caption length: 64.0 characters

2. TEXT LENGTH STATISTICS
   Min length: 2 characters
   Max length: 703 characters
   25th percentile: 37.0 characters
   75th percentile: 94.0 characters
   Standard deviation: 36.9 characters

3. WORD COUNT STATISTICS
   Average words per caption: 12.6
   Median words per caption: 12.0
   Min words: 1
   Max words: 32

4. COMPLETE EMOTION DI

## EDA Insights & Implications for Model Training

Our exploratory data analysis provided several insights that directly inform model training strategies:

### 1. Text Length Diversity
The dataset contains captions with a wide range of lengths. To accommodate this variability, we set a maximum tokenization length of `128`, which effectively captures the majority of examples while truncating outliers.

### 2. Class Imbalance Challenge
EDA revealed a **severe class imbalance**:
- The most represented emotion (`neutral`) has **12,823 samples**, making up **35.3%** of the dataset.
- The least represented emotions (e.g., `grief`, `pride`) have fewer than **50 samples**.
- The resulting **imbalance ratio** is as high as **328.8:1**.

### 3. High Caption Uniqueness
Over **95%** of the captions are unique, indicating high linguistic diversity. This reduces overfitting risk and supports rich language modeling.

### 4. Strong Data Quality
The dataset has:
- No missing captions
- Very few extremely short (<10 chars) or long (>200 chars) examples

This ensures the model receives clean, well-formed inputs without the need for heavy preprocessing.

### 5. Implications for Training
Given the above findings:
- **Balancing the dataset** is essential to ensure fair learning across all emotion classes.
- The data is suitable for training a GPT-2 model with minimal cleaning.
- **Fine-tuning strategies** should monitor performance on underrepresented emotions.
- Emotion-specific evaluation metrics, such as polarity alignment, will be used to assess the effectiveness of mood-conditioned generation.


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Addressing Dataset Class Imbalance</strong></h2>
  <p style="color:#333333;">Balancing the dataset is essential to ensure fair learning across all emotion classes.</p>
</div>

In [7]:
def balance_dataset_for_training(df, max_samples_per_emotion=2000, min_samples_per_emotion=50):
    """
    Address class imbalance by limiting over-represented emotions and 
    ensuring minimum representation for under-represented ones.
    
    Args:
        df (pd.DataFrame): DataFrame with mood and caption columns
        max_samples_per_emotion (int): Maximum samples per emotion
        min_samples_per_emotion (int): Minimum samples per emotion
        
    Returns:
        pd.DataFrame: Balanced dataset
    """
    print("Addressing class imbalance...")
    
    balanced_dfs = []
    emotion_counts = df['mood'].value_counts()
    
    for emotion in emotion_counts.index:
        emotion_data = df[df['mood'] == emotion]
        current_count = len(emotion_data)
        
        if current_count > max_samples_per_emotion:
            # Downsample over-represented emotions
            sampled_data = emotion_data.sample(n=max_samples_per_emotion, random_state=42)
            print(f"   {emotion}: {current_count} -> {max_samples_per_emotion} (downsampled)")
        elif current_count < min_samples_per_emotion:
            # Upsample under-represented emotions (with replacement)
            sampled_data = emotion_data.sample(n=min_samples_per_emotion, replace=True, random_state=42)
            print(f"   {emotion}: {current_count} -> {min_samples_per_emotion} (upsampled)")
        else:
            # Keep as is
            sampled_data = emotion_data
            print(f"   {emotion}: {current_count} (unchanged)")
            
        balanced_dfs.append(sampled_data)
    
    balanced_df = pd.concat(balanced_dfs, ignore_index=True)
    
    # Shuffle the balanced dataset
    balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\nBalanced dataset size: {len(balanced_df)} (was {len(df)})")
    print("New emotion distribution:")
    print(balanced_df['mood'].value_counts().head(10))
    
    return balanced_df

In [8]:
def create_training_dataset(df, format_type="structured"):
    """
    Convert dataframe to HuggingFace Dataset with proper formatting.
    
    Args:
        df (pd.DataFrame): DataFrame with mood and caption columns
        format_type (str): "structured" or "simple" formatting
        
    Returns:
        Dataset: HuggingFace dataset ready for training
    """
    print("Creating training dataset...")
    
    if format_type == "structured":
        # Create improved structured prompt format with explicit task instruction
        df["text"] = df.apply(lambda row: f"Generate a {row['mood']} caption: {row['caption']}<|endoftext|>", axis=1)
    else:
        # Simple format: "X: Y"
        df["text"] = df.apply(lambda row: f"{row['mood']}: {row['caption']}<|endoftext|>", axis=1)
    
    # Convert to HuggingFace dataset
    dataset = Dataset.from_pandas(df[["text"]])
    print(f"Created dataset with {len(dataset)} training examples")
    
    return dataset

In [9]:
def tokenize_dataset(dataset, model_name="gpt2", max_length=128):
    """
    Tokenize dataset for GPT-2 training.
    
    Args:
        dataset (Dataset): HuggingFace dataset to tokenize
        model_name (str): Model name for tokenizer
        max_length (int): Maximum sequence length
        
    Returns:
        Dataset: Tokenized dataset ready for training
    """
    print("Tokenizing dataset...")
    
    # Initialize tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    # Tokenization function
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", 
                        truncation=True, max_length=max_length)
    
    # Apply tokenization
    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    print(f"Tokenized {len(tokenized_dataset)} examples")
    
    return tokenized_dataset, tokenizer

In [10]:
# Address class imbalance for better training
print("=" * 50)
print("ADDRESSING CLASS IMBALANCE")
print("=" * 50)

# Apply balancing to improve training performance
balanced_df = balance_dataset_for_training(df, max_samples_per_emotion=1500, min_samples_per_emotion=100)

# Update the dataset creation to use balanced data
print("\n" + "=" * 50)
print("Creating balanced training dataset...")
dataset = create_training_dataset(balanced_df, format_type="structured")
tokenized_dataset, tokenizer = tokenize_dataset(dataset)

print("\nBalanced data preparation complete.")

ADDRESSING CLASS IMBALANCE
Addressing class imbalance...
   neutral: 12823 -> 1500 (downsampled)
   admiration: 2710 -> 1500 (downsampled)
   approval: 1873 -> 1500 (downsampled)
   gratitude: 1857 -> 1500 (downsampled)
   amusement: 1652 -> 1500 (downsampled)
   annoyance: 1451 (unchanged)
   love: 1427 (unchanged)
   disapproval: 1402 (unchanged)
   curiosity: 1389 (unchanged)
   anger: 1025 (unchanged)
   optimism: 861 (unchanged)
   confusion: 858 (unchanged)
   joy: 853 (unchanged)
   sadness: 817 (unchanged)
   surprise: 720 (unchanged)
   disappointment: 709 (unchanged)
   caring: 649 (unchanged)
   realization: 586 (unchanged)
   excitement: 510 (unchanged)
   disgust: 498 (unchanged)
   fear: 430 (unchanged)
   desire: 389 (unchanged)
   remorse: 353 (unchanged)
   embarrassment: 203 (unchanged)
   relief: 88 -> 100 (upsampled)
   nervousness: 85 -> 100 (upsampled)
   pride: 51 -> 100 (upsampled)
   grief: 39 -> 100 (upsampled)

Balanced dataset size: 23030 (was 36308)
New emo

Map: 100%|██████████| 23030/23030 [00:07<00:00, 3113.59 examples/s]

Tokenized 23030 examples

Balanced data preparation complete.


## Class Imbalance Successfully Addressed

### Balancing Strategy Applied:

Our `balance_dataset_for_training()` function implements a **hybrid sampling approach**:

1. **Downsampling**: Limit over-represented emotions to **1,500 samples max**
   - Prevents `neutral` from dominating training (was 12,823 → now 1,500)
   - Maintains data quality by keeping diverse examples

2. **Upsampling**: Ensure under-represented emotions have **100 samples min**
   - Uses replacement sampling to boost rare emotions
   - Gives every emotion fair learning opportunity

3. **Shuffling**: Randomize the balanced dataset to prevent order bias

### Impact on Training Quality:

**Before Balancing:**
- Extreme bias toward neutral content
- Poor learning for rare emotions like `grief`, `pride`, `relief`
- High risk of mode collapse and repetitive generation

**After Balancing:**
- **~15:1** maximum imbalance ratio (much more manageable)
- All emotions have meaningful representation
- Better emotion conditioning expected
- More stable training dynamics

### Expected Model Improvements:

• **Diverse Emotion Generation**: Model can now learn patterns for all 28 emotions  
• **Reduced Bias**: No single emotion dominates the training signal  
• **Better Evaluation**: Fairer performance assessment across emotion categories  
• **Stable Training**: Balanced gradients prevent training instability


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Baseline Experiments - Zero-Shot</strong></h2>
  <p style="color:#333333;">Setup the benchmark for model tuning performance comparison.</p>
</div>

## Baseline Experiments: Prepare for Zero-Shot vs Fine-Tuned Comparison

### Data-Driven Emotion Selection (Post-Balancing)

Now that we understand our training data through EDA and have applied dataset balancing, we can make informed decisions about baseline testing:

**Based on Post-Balancing Distribution:**
- **High-frequency emotions** (1,500 samples): `neutral`, `admiration`, `approval`, `gratitude`, `amusement`
- **Medium-frequency emotions** (400-1,500 samples): `annoyance`, `love`, `disapproval`, `curiosity`, `anger`, `optimism`, `confusion`, `joy`, `sadness`, `surprise`, `disappointment`, `caring`, `realization`, `excitement`, `disgust`, `fear`, `desire`, `remorse`, `embarrassment`  
- **Low-frequency emotions** (100 samples, upsampled): `relief`, `nervousness`, `pride`, `grief`

**Baseline Strategy:**
We'll test a **representative sample** spanning the balanced frequency spectrum to understand:
1. How well GPT-2 handles emotions with maximum representation (1,500 samples)
2. Performance on naturally medium-frequency emotions that weren't resampled
3. Challenges with artificially upsampled low-frequency emotions

**Key Insight:** Post-balancing, we expect more consistent performance across emotions since the extreme imbalance (328:1 ratio) has been reduced to a manageable 15:1 ratio. This gives us realistic baselines for comparison after fine-tuning on a more balanced dataset.

## Understanding Sentiment Polarity for Evaluation

**Polarity** is a key metric in sentiment analysis that measures the emotional tone of text on a scale from -1 to +1:

- **Positive Values (+0.1 to +1.0)**: Indicate positive sentiment (joy, happiness, love, excitement)
- **Negative Values (-0.1 to -1.0)**: Indicate negative sentiment (sadness, anger, fear, disgust)  
- **Neutral Values (~0.0)**: Indicate neutral or objective text

**Why Polarity Matters for Our Project:**

In our mood-conditioned caption generation task, polarity serves as a quantitative measure to evaluate whether generated captions align with the intended emotional mood. For example:

- A caption generated for mood "joy" should ideally have a positive polarity (>0.1)
- A caption for mood "sadness" should have a negative polarity (<-0.1)
- A caption for mood "anger" should also have negative polarity

High-frequency emotions should show better improvement after fine-tuning, while low-frequency emotions may remain challenging even after training.

**Our Data-Driven Evaluation Strategy:**

We use TextBlob's sentiment analysis to calculate polarity scores, which helps us:
1. **Establish informed baselines** after understanding our training data through EDA
2. **Quantitatively compare** fine-tuned vs baseline models on relevant emotions
3. **Measure improvement** in emotional alignment across different frequency categories


In [11]:
# Initialize zero-shot GPT-2 pipeline for baseline testing
print("BASELINE POLARITY TESTING")
print("=" * 60)
print("Testing emotions selected based on post-balancing distribution...")

zero_shot_generator = pipeline("text-generation", model="gpt2", tokenizer="gpt2")

# Post-balancing emotion selection spanning balanced frequency spectrum
test_moods = {
    "High-frequency": ["neutral", "admiration", "approval", "gratitude"],  # 1,500 samples each
    "Medium-frequency": ["joy", "sadness", "anger", "love", "fear"],       # 400-1,500 samples  
    "Low-frequency": ["grief", "pride", "relief"]                          # 100 samples (upsampled)
}

print("\nBASELINE RESULTS (Zero-Shot GPT-2) - Post-Balancing Selection:")
print("=" * 70)

all_results = {}
for category, emotions in test_moods.items():
    print(f"\n{category.upper()} EMOTIONS:")
    print("-" * 50)
    
    for mood in emotions:
        # Use more natural prompts that GPT-2 can better understand
        if mood == "neutral":
            prompt = "Caption: "
        else:
            prompt = f"I feel {mood}. Caption: "
        
        output = zero_shot_generator(prompt, max_new_tokens=20, num_return_sequences=1, 
                                    do_sample=True, temperature=0.8, top_p=0.9,
                                    pad_token_id=50256)
        generated_text = output[0]["generated_text"].replace(prompt, "").strip()
        
        # Calculate sentiment polarity using TextBlob
        polarity = TextBlob(generated_text).sentiment.polarity
        all_results[mood] = {"text": generated_text, "polarity": polarity, "category": category}
        
        print(f"Mood: {mood}")
        print(f"Generated: {generated_text}")
        print(f"Polarity: {polarity:.3f}")
        print("-" * 30)

# Calculate average polarity by category
for category in test_moods.keys():
    category_results = [r for r in all_results.values() if r["category"] == category]
    avg_polarity = sum(r["polarity"] for r in category_results) / len(category_results)
    print(f"\n{category} Average Polarity: {avg_polarity:.3f}")

overall_avg = sum(r["polarity"] for r in all_results.values()) / len(all_results)
print(f"\nOverall Baseline Average Polarity: {overall_avg:.3f}")
print("Baseline established with balanced dataset emotion selection.")

BASELINE POLARITY TESTING
Testing emotions selected based on post-balancing distribution...


Device set to use cuda:0



BASELINE RESULTS (Zero-Shot GPT-2) - Post-Balancing Selection:

HIGH-FREQUENCY EMOTIONS:
--------------------------------------------------
Mood: neutral
Generated: A photo from a young woman holding a baby.
The first thing I did was make sure
Polarity: 0.283
------------------------------
Mood: admiration
Generated: I feel admiration.
What a great thing to see from the White House. There are some
Polarity: 0.400
------------------------------
Mood: approval
Generated:  Trump's first 100 days in office. (Trent Lott/The Washington Post
Polarity: 0.250
------------------------------
Mood: gratitude
Generated: I feel gratitude.  We need to see the same. We need to see the same
Polarity: 0.000
------------------------------

MEDIUM-FREQUENCY EMOTIONS:
--------------------------------------------------
Mood: joy
Generated: A young boy holds his newborn baby after being rushed to the hospital after the shooting in Charleston,
Polarity: 0.100
------------------------------
Mood: sadness
Gen

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Mood: grief
Generated: (CNN) - A mother who was raped by her son and her husband in their home in
Polarity: 0.000
------------------------------
Mood: pride
Generated: (Alfred A. Thompson)
It's hard to imagine a more powerful feeling than
Polarity: 0.169
------------------------------
Mood: relief
Generated: A little girl named Charlotte was killed and her body found in a wooded area in the woods
Polarity: -0.194
------------------------------

High-frequency Average Polarity: 0.233

Medium-frequency Average Polarity: 0.001

Low-frequency Average Polarity: -0.008

Overall Baseline Average Polarity: 0.076
Baseline established with balanced dataset emotion selection.


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Meme Formatting & Data Cleaning</strong></h2>
  <p style="color:#333333;">Transform captions into meme-style format with structured TOP/BOTTOM layout.</p>
</div>

## Meme Formatting & Data Cleaning Strategy

### **Problem Identified:**
The original GoEmotions captions contain **long social media text** like:
- *"I'm sorry that I'm a little late to the party but I'm still here."*
- *"Pretty sure I've seen this. He swings away with the harness he is wearing. Still..."*

But **meme-style captions** need **short, punchy format**:
- **TOP TEXT**: "WHEN YOU'RE LATE"  
- **BOTTOM TEXT**: "BUT STILL SHOW UP"

### **Solution: Meme Formatting Pipeline**

The `create_meme_training_dataset()` function implements meme formatting by:

1. **Text Cleaning**: Remove URLs, mentions, hashtags, excessive punctuation
2. **Length Limiting**: Cap cleaned captions at 8 words maximum  
3. **Format Conversion**: Split longer text into TOP/BOTTOM meme format
4. **Case Normalization**: Convert to ALL CAPS (standard meme style)
5. **Training Format**: Use structured prompts like `"Generate a joy meme:\nTOP: WHEN YOU...\nBOTTOM: ..."`

### **Expected Benefits:**
- **Meme-style captions** suitable for overlay rendering
- **Structured format** matching the `3a_meme_text_rendering.ipynb` pipeline
- **Better emotion alignment** with cleaned, focused text
- **Production-ready** output that fits meme constraints

In [12]:
def clean_text_for_memes(text, max_words=8):
    """
    Clean and prepare text for meme-style captions.
    
    Args:
        text (str): Original text
        max_words (int): Maximum words per caption
        
    Returns:
        str: Cleaned meme-appropriate text
    """
    import re
    
    # Remove URLs, mentions, hashtags
    text = re.sub(r'http\S+|www\S+|@\w+|#\w+', '', text)
    
    # Remove extra whitespace and newlines
    text = ' '.join(text.split())
    
    # Remove quotes and special characters, keep basic punctuation
    text = re.sub(r'["""''`]', '', text)
    text = re.sub(r'[^\w\s,.!?-]', '', text)
    
    # Split into words and limit length
    words = text.split()
    if len(words) > max_words:
        # Take first part that makes sense
        text = ' '.join(words[:max_words])
    
    # Ensure it ends properly
    if text and not text[-1] in '.!?':
        if len(words) > max_words:
            text += '...'
    
    return text.strip().upper()

def split_into_meme_format(text):
    """
    Split longer text into TOP and BOTTOM meme format.
    
    Args:
        text (str): Input text
        
    Returns:
        tuple: (top_text, bottom_text) or (text, "") if short
    """
    words = text.split()
    
    if len(words) <= 4:
        return text, ""
    
    # Try to split at natural break points
    split_points = []
    for i, word in enumerate(words):
        if word.lower() in ['and', 'but', 'so', 'when', 'then', 'because']:
            split_points.append(i)
    
    # Use middle split if no natural break found
    if not split_points:
        split_point = len(words) // 2
    else:
        # Use split point closest to middle
        split_point = min(split_points, key=lambda x: abs(x - len(words) // 2))
    
    top_text = ' '.join(words[:split_point])
    bottom_text = ' '.join(words[split_point:])
    
    return top_text, bottom_text

def create_meme_training_dataset(df, format_type="meme_format"):
    """
    Convert dataframe to HuggingFace Dataset with meme formatting.
    
    Args:
        df (pd.DataFrame): DataFrame with mood and caption columns
        format_type (str): "meme_format", "structured", or "simple"
        
    Returns:
        Dataset: HuggingFace dataset ready for training meme-style captions
    """
    print("Creating meme-formatted training dataset...")
    
    if format_type == "meme_format":
        # Clean and prepare captions for meme formatting
        print("Applying meme formatting to captions...")
        
        # Clean the captions
        df["clean_caption"] = df["caption"].apply(clean_text_for_memes)
        
        # Filter out very short or empty captions
        df = df[df["clean_caption"].str.len() > 3].copy()
        print(f"After meme formatting: {len(df)} cleaned captions remain")
        
        # Show some examples of before/after meme formatting
        print("\nMeme formatting examples:")
        for i in range(min(3, len(df))):
            original = df.iloc[i]["caption"]
            cleaned = df.iloc[i]["clean_caption"]
            print(f"  Original caption: '{original[:50]}...'")
            print(f"  Cleaned caption:  '{cleaned}'")
            print()
        
        # Create meme-style caption training data
        training_texts = []
        for _, row in df.iterrows():
            mood = row['mood']
            caption = row['clean_caption']
            
            # Split into top/bottom if long enough
            top_text, bottom_text = split_into_meme_format(caption)
            
            if bottom_text:
                # Two-part meme format
                meme_text = f"TOP: {top_text}\nBOTTOM: {bottom_text}"
            else:
                # Single-part meme format (will be TOP only)
                meme_text = f"TOP: {top_text}\nBOTTOM: "
            
            # Create training prompt
            training_prompt = f"Generate a {mood} meme:\n{meme_text}<|endoftext|>"
            training_texts.append(training_prompt)
        
        # Create DataFrame for dataset conversion
        training_df = pd.DataFrame({"text": training_texts})
        
        # Show example of final meme-style caption training format
        print("Meme-style caption training format example:")
        print(training_texts[0])
        print()
        
    else:
        # Fall back to original function
        return create_training_dataset(df, format_type)
    
    # Convert to HuggingFace dataset
    dataset = Dataset.from_pandas(training_df)
    print(f"Created meme-formatted dataset with {len(dataset)} training examples")
    
    return dataset

In [13]:
# Test the meme formatting on a small sample
print("TESTING MEME FORMATTING")
print("=" * 50)

# Test with a small sample first
test_sample = balanced_df.sample(n=5, random_state=42)

print("BEFORE MEME FORMATTING:")
print("-" * 30)
for i, row in test_sample.iterrows():
    print(f"{row['mood']}: '{row['caption']}'")

print("\nAFTER MEME FORMATTING:")
print("-" * 30)

# Apply meme formatting to see the difference
for i, row in test_sample.iterrows():
    original = row['caption']
    cleaned = clean_text_for_memes(original, max_words=8)
    top_text, bottom_text = split_into_meme_format(cleaned)
    
    print(f"{row['mood']}:")
    print(f"  Original: '{original}'")
    print(f"  Cleaned:  '{cleaned}'")
    if bottom_text:
        print(f"  Format:   TOP: '{top_text}' | BOTTOM: '{bottom_text}'")
    else:
        print(f"  Format:   TOP: '{top_text}' | BOTTOM: (empty)")
    print()

print("Meme formatting ready, we can use create_meme_training_dataset() for training.")

TESTING MEME FORMATTING
BEFORE MEME FORMATTING:
------------------------------
annoyance: 'Israel is only demonized because they DEFEND themselves from their neighbors who OPENLY admit they are dedicated to the destruction of Israel and the [RELIGION] people.'
fear: 'This is scary.'
neutral: 'You didn't tell me there was a gas lesk!'
amusement: '[NAME] made that lmao'
approval: 'Trolls are okay in this sub but let's be more serious here'

AFTER MEME FORMATTING:
------------------------------
annoyance:
  Original: 'Israel is only demonized because they DEFEND themselves from their neighbors who OPENLY admit they are dedicated to the destruction of Israel and the [RELIGION] people.'
  Cleaned:  'ISRAEL IS ONLY DEMONIZED BECAUSE THEY DEFEND THEMSELVES...'
  Format:   TOP: 'ISRAEL IS ONLY DEMONIZED' | BOTTOM: 'BECAUSE THEY DEFEND THEMSELVES...'

fear:
  Original: 'This is scary.'
  Cleaned:  'THIS IS SCARY.'
  Format:   TOP: 'THIS IS SCARY.' | BOTTOM: (empty)

neutral:
  Original: 'You di

In [14]:
# Create meme-formatted version of the training dataset
print("Using the same balanced_df, but with meme formatting...")
meme_dataset = create_meme_training_dataset(balanced_df, format_type="meme_format")
meme_tokenized_dataset, meme_tokenizer = tokenize_dataset(meme_dataset)

print("\nDATASET COMPARISON:")
print("-" * 40)
print(f"Original structured dataset: {len(tokenized_dataset)} examples")
print(f"Meme-formatted dataset:     {len(meme_tokenized_dataset)} examples")

print("\nTRAINING FORMAT COMPARISON:")
print("-" * 40)
print("Original Format Example:")
print(dataset[0]['text'][:100] + "..." if len(dataset[0]['text']) > 100 else dataset[0]['text'])

print("\nMeme-Style Caption Format Example:")
print(meme_dataset[0]['text'][:100] + "..." if len(meme_dataset[0]['text']) > 100 else meme_dataset[0]['text'])

# For the training section, we now use meme-formatted dataset
training_dataset = meme_tokenized_dataset
training_tokenizer = meme_tokenizer

Using the same balanced_df, but with meme formatting...
Creating meme-formatted training dataset...
Applying meme formatting to captions...
After meme formatting: 23026 cleaned captions remain

Meme formatting examples:
  Original caption: 'Yeah who knows how many indictments [NAME] is gonn...'
  Cleaned caption:  'YEAH WHO KNOWS HOW MANY INDICTMENTS NAME IS...'

  Original caption: 'just keep swimming further off that deep end...'
  Cleaned caption:  'JUST KEEP SWIMMING FURTHER OFF THAT DEEP END'

  Original caption: '>US isn't a de facto dictatorship Not sure anyone ...'
  Cleaned caption:  'US ISNT A DE FACTO DICTATORSHIP NOT SURE...'

Meme-style caption training format example:
Generate a confusion meme:
TOP: YEAH WHO KNOWS HOW
BOTTOM: MANY INDICTMENTS NAME IS...<|endoftext|>

Created meme-formatted dataset with 23026 training examples
Tokenizing dataset...


Map: 100%|██████████| 23026/23026 [00:06<00:00, 3371.87 examples/s]

Tokenized 23026 examples

DATASET COMPARISON:
----------------------------------------
Original structured dataset: 23030 examples
Meme-formatted dataset:     23026 examples

TRAINING FORMAT COMPARISON:
----------------------------------------
Original Format Example:
Generate a confusion caption: Yeah who knows how many indictments [NAME] is gonna rack up retroactiv...

Meme-Style Caption Format Example:
Generate a confusion meme:
TOP: YEAH WHO KNOWS HOW
BOTTOM: MANY INDICTMENTS NAME IS...<|endoftext|>


<div style="background-color:#e6f2ff; border-left:8px solid #0059b3; padding:20px; margin:20px 0;">
  <h2 style="color:#003366;"><strong>Model Fine-Tuning</strong></h2>
  <p style="color:#333333;">Train GPT-2 on our balanced, meme-formatted dataset to generate emotionally aligned meme-style captions. This section implements the fine-tuning pipeline with checkpoint resume support and optimized training parameters for mood-conditioned text generation.</p>
</div>

In [15]:
def fine_tune_gpt2(tokenized_dataset, tokenizer, output_dir="../models/gpt2-mood-caption-v2", 
                   epochs=5, batch_size=4, learning_rate=2e-5, resume_from_checkpoint=None):
    """
    Fine-tune GPT-2 model for mood-conditioned caption generation.
    
    Args:
        tokenized_dataset (Dataset): Tokenized training dataset
        tokenizer: GPT-2 tokenizer
        output_dir (str): Directory to save the fine-tuned model
        epochs (int): Number of training epochs
        batch_size (int): Training batch size
        learning_rate (float): Learning rate for training
        resume_from_checkpoint (str, optional): Path to checkpoint to resume training from
        
    Returns:
        Trainer: Trained model trainer object
    """
    print("Starting GPT-2 Fine-Tuning...")
    
    # Load base GPT-2 model
    model = GPT2LMHeadModel.from_pretrained("gpt2")
    print("Loaded base GPT-2 model")
    
    # Data collator for language modeling
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    
    # Training arguments - optimized for better fine-tuning
    training_args = TrainingArguments(
        output_dir=output_dir,
        report_to="none",  # Disable wandb logging
        per_device_train_batch_size=batch_size,
        num_train_epochs=epochs,
        learning_rate=learning_rate,
        warmup_steps=200,  # Learning rate warmup for stability
        lr_scheduler_type="cosine",  # Cosine learning rate decay
        save_steps=1000,
        save_total_limit=3,
        logging_steps=50,  # More frequent logging
        # evaluation_strategy="steps" if len(tokenized_dataset) > 10000 else "no",
        # eval_steps=1000 if len(tokenized_dataset) > 10000 else None,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),  # Use mixed precision if GPU available
        dataloader_drop_last=True,
        gradient_accumulation_steps=2,  # Effective batch size = 4 * 2 = 8
        adam_epsilon=1e-8,  # More stable optimizer
        max_grad_norm=1.0,  # Gradient clipping
    )
    
    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    
    print(f"Training Configuration:")
    print(f"   Dataset size: {len(tokenized_dataset)} examples")
    print(f"   Epochs: {epochs}")
    print(f"   Batch size: {batch_size}")
    print(f"   Learning rate: {learning_rate}")
    print(f"   Output directory: {output_dir}")
    print(f"   Using GPU: {torch.cuda.is_available()}")
    print(f"   Resume from checkpoint: {resume_from_checkpoint if resume_from_checkpoint else 'Starting fresh'}")
    
    # Start training
    print("\nStarting training...")
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)
    
    # Ensure models directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Save the fine-tuned model
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    print(f"\nFine-tuning complete! Model saved to {output_dir}")
    return trainer

In [16]:
def get_checkpoint_or_none(output_dir, subdirectory="checkpoints"):
    """
    Get checkpoint or None with consistent logging.
    
    This function eliminates repeated checkpoint resume logic by providing
    a single, reusable function for checkpoint detection and resume logic.
    
    Args:
        output_dir (str): Main output directory (e.g., "../models/gpt2-mood-caption-v2")
        subdirectory (str): Checkpoint subdirectory name (default: "checkpoints")
        
    Returns:
        str or None: Path to latest checkpoint, or None if no checkpoints found
    """
    if not os.path.exists(output_dir):
        print("No output directory found, cannot search for checkpoints.")
        return None
    
    # List all entries in the output directory that start with 'checkpoint-' and are directories
    checkpoint_files = [f for f in os.listdir(output_dir) 
                        if f.startswith('checkpoint-') and os.path.isdir(os.path.join(output_dir, f))]
    
    if not checkpoint_files:
        print("No existing checkpoints found — starting from scratch.")
        return None
        
    # Sort by checkpoint number to find the latest one
    checkpoint_files.sort(key=lambda x: int(x.split('-')[1]))
    latest_checkpoint = os.path.join(output_dir, checkpoint_files[-1])
    
    print(f"Resuming training from checkpoint: {latest_checkpoint}")
    return latest_checkpoint

In [17]:
# TRAINING EXECUTION: Using Meme-Formatted Dataset

# Configuration
output_dir = "../models/gpt2-mood-caption-v2"

# 🔧 DRY Fix: Use helper function instead of repeated checkpoint logic
resume_checkpoint = get_checkpoint_or_none(output_dir)

print("\n" + "=" * 70)

# Start training with meme-formatted dataset
try:
    trainer = fine_tune_gpt2(
        tokenized_dataset=training_dataset,  # Using meme-formatted dataset
        tokenizer=training_tokenizer,        # Using meme-formatted tokenizer
        output_dir=output_dir,
        epochs=5,  # More epochs for better learning
        batch_size=4,
        learning_rate=2e-5,  # Lower learning rate for stability
        resume_from_checkpoint=resume_checkpoint  # Now using DRY helper function
    )
    
    print("\n" + "=" * 60)
    print("TRAINING COMPLETED SUCCESSFULLY!")
    print("Model trained on meme-formatted dataset!")
    print("Ready for meme-style caption generation.")
    print("=" * 60)
    
except KeyboardInterrupt:
    print("\n" + "!" * 60)
    print("WARNING: Training was interrupted.")
    print("Training was interrupted. You can resume later.")
    print("!" * 60)
    
except Exception as e:
    print("\n" + "!" * 60)
    print("ERROR: Training failed.")
    print(f"Training failed with error: {e}")
    print("Check the checkpoint directory for partial progress.")
    print("!" * 60)
    raise e

Resuming training from checkpoint: ../models/gpt2-mood-caption-v2/checkpoint-14390

Starting GPT-2 Fine-Tuning...
Loaded base GPT-2 model
Training Configuration:
   Dataset size: 23026 examples
   Epochs: 5
   Batch size: 4
   Learning rate: 2e-05
   Output directory: ../models/gpt2-mood-caption-v2
   Using GPU: True
   Resume from checkpoint: ../models/gpt2-mood-caption-v2/checkpoint-14390

Starting training...


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Step,Training Loss



Fine-tuning complete! Model saved to ../models/gpt2-mood-caption-v2

TRAINING COMPLETED SUCCESSFULLY!
Model trained on meme-formatted dataset!
Ready for meme-style caption generation.


## Summary & Next Steps

### What We Accomplished

1. **Defined Clear Objectives**: Established controlled text generation as our NLP task with emotional alignment goals
2. **Literature Review**: Justified GPT-2 selection based on research in controlled generation and meme captioning
3. **Baseline Establishment**: Tested zero-shot GPT-2 performance for comparison with fine-tuned model
4. **Modular Implementation**: Created reusable functions for data preparation, tokenization, and model training
5. **Model Training**: Successfully fine-tuned GPT-2 on GoEmotions dataset for mood-conditioned caption generation

### Model Outputs

- **Fine-tuned Model**: `../models/gpt2-mood-caption-v2/` (saved to main project models folder)
- **Training Data**: `../data/mood_captions_goemotions.csv` (saved to main project data folder)
- **Baseline Results**: Stored for comparison in next notebook

### Next Steps

The next notebook (`2b_caption_generation.ipynb`) will focus on:
- Loading and using the fine-tuned model for inference
- Comprehensive evaluation metrics (polarity analysis, semantic similarity)
- Comparison between baseline and fine-tuned performance
- Production-ready caption generation functions
- Integration-ready code for the cartoonization pipeline

### Reusability

All functions are modular and documented to support:
- Easy integration into larger systems
- Reproducible model training and inference
- Adaptation for different datasets or model architectures
- Deployment in production environments
